Libraries

In [1]:
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn
%pip install plotly
%pip install nltk
%pip install langdetect
%pip install wordcloud
%pip install transformers
%pip install torch
%pip install tensorflow
%pip install stopwords


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1


In [2]:


import pandas as pd
import numpy as np 

# Visualization 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.graph_objects as go 
import plotly.express as px 
from plotly.subplots import make_subplots

# Text processing 
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory
import warnings 
warnings.filterwarnings('ignore')

In [3]:
# this helps to avoid libraries clashing with each other (e.g some libraries may need to be updated)
warnings.filterwarnings('ignore')

# set random seed for reproducibility
np.random.seed(42)
DetectorFactory.seed = 42

In [4]:
nltk.download('punkt', quiet=True) # pre-trained model that helps nltk to split text into sentences
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [5]:
dataset = '../data/raw/raw_reviews.csv'

In [6]:
data = pd.read_csv(dataset)

In [7]:
data.head()

,review_id,product_category,timestamp,country,rating,review,sentiment
0,1,Books,2023-01-01,GB,3,"Solid build, attractive design, works as adver...",positive
1,2,Toys,2023-01-01,DE,5,Ich liebe dieses Produkt! ⭐⭐⭐,positive
2,3,Beauty,2023-01-01,AU,3,Three stars — meets THE minimum expectations.,neutral
3,4,Toys,2023-01-01,US,5,"Solid build, attractive design, works as adver...",positive
4,5,Electronics,2023-01-01,CA,2,Broken on arrival. Return process was a NIGHTM...,negative


In [8]:
data.shape

(120000, 7)

In [9]:
data.sentiment.value_counts()

sentiment
positive    81591
negative    19233
neutral     19176
Name: count, dtype: int64

In [10]:
positive = data[data['sentiment']== 'positive'].sample(n=10000, random_state=42)
negative = data[data['sentiment']== 'negative'].sample(n=10000, random_state=42)
neutral = data[data['sentiment']== 'neutral'].sample(n=10000, random_state=42)

# i used concat to stack the dataset insted of merge that will just join them together 
df= pd.concat([positive, negative, neutral])


# this is to reshuffle the stack of data randomly 
df= df.sample(frac=1, random_state=42).reset_index(drop=True)

In [11]:
df.shape

(30000, 7)

In [12]:
df.columns.to_list()

['review_id',
 'product_category',
 'timestamp',
 'country',
 'rating',
 'review',
 'sentiment']

In [13]:
df.describe(include='all')

,review_id,product_category,timestamp,country,rating,review,sentiment
count,30000.000000,30000,30000,30000,30000.000000,30000,30000
unique,NaN,8,1095,14,NaN,2349,3
top,NaN,Electronics,2023-03-27,GB,NaN,C'est correct pour le prix. lol,positive
freq,NaN,3858,45,5347,NaN,393,10000
mean,59893.232800,NaN,NaN,NaN,3.188967,NaN,NaN
std,34532.815444,NaN,NaN,NaN,1.266667,NaN,NaN
min,3.000000,NaN,NaN,NaN,1.000000,NaN,NaN
25%,30210.750000,NaN,NaN,NaN,2.000000,NaN,NaN
50%,59650.500000,NaN,NaN,NaN,3.000000,NaN,NaN
75%,89741.750000,NaN,NaN,NaN,4.000000,NaN,NaN


In [14]:
# print out all the categorical columns that identify as object with its unique values
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f" {col}: {df[col].nunique()} unique values")

 product_category: 8 unique values
 timestamp: 1095 unique values
 country: 14 unique values
 review: 2349 unique values
 sentiment: 3 unique values


In [15]:
sentiment_counts = df['sentiment'].value_counts()
sentiment_percent = (sentiment_counts / len(df))*100

In [16]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Sentiment Distribution', 'Sentiment Percentage'),
    specs= [[{"type": "bar"}, {"type": "pie"}]]
)

#Bar Chart
fig.add_trace(
    go.Bar(
        x=sentiment_counts.index,
        y=sentiment_counts.values,
        marker_color=['green', 'blue', 'red'],
        text=sentiment_counts.values,
        textposition='auto', 
        name='Count'
    ),
    row=1, col=1
)

#Pie Chart
fig.add_trace(
    go.Pie(
        labels=sentiment_counts.index,
        values=sentiment_counts.values,
        marker_colors=['green', 'blue', 'red'],
        textinfo='percent+label',
        hole=0.3
    ),
    row=1, col=2
) 

fig.update_layout(
    title_text = "Sentiment Distribution Analysis",
    height=400,
    showlegend=False
)

fig.show()

In [17]:
for sentiment, count in sentiment_counts.items():
    print(f" {sentiment.capitalize()}: {count} reviews ({sentiment_percent[sentiment]:.1f}%)")

 Positive: 10000 reviews (33.3%)
 Neutral: 10000 reviews (33.3%)
 Negative: 10000 reviews (33.3%)


In [18]:
rating_counts = df['rating'].value_counts().sort_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=rating_counts.index,
    y=rating_counts.values,
    marker_color='skyblue',
    text=rating_counts.values,
    textposition='auto'
    
))

fig.update_layout(
    title_text= "Rating Distribution",
    xaxis_title="Rating (Stars)",
    yaxis_title="Number of Reviews",
    height=400

)

fig.show()
              


In [19]:
rating_sentiments= pd.crosstab(df['rating'], df['sentiment'])
rating_sentiments_pct = pd.crosstab(df['rating'], df['sentiment'], normalize='index')*100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Rating vs Sentiment (Count)', 'Rating vs Sentiment (Percentage)'),
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Count plot
for sentiment in ['positive', 'neutral', 'negative']:
    if sentiment in rating_sentiments.columns:
        fig.add_trace(
            go.Bar(
                name=sentiment.capitalize(),
                x=rating_sentiments.index,
                y=rating_sentiments[sentiment].values,
                marker_color=['green', 'blue', 'red'][['positive', 'neutral', 'negative'].index(sentiment)]
            ),
            row=1, col=1
        )
        
# Percentage plot

for sentiment in ['positive', 'neutral', 'negative']:
    if sentiment in rating_sentiments_pct.columns:
        fig.add_trace(
            go.Bar(
                name=sentiment.capitalize(),
                x=rating_sentiments_pct.index,
                y=rating_sentiments_pct[sentiment].values,
                marker_color=['green', 'blue', 'red'][['positive', 'neutral', 'negative'].index(sentiment)],
                showlegend=False
            ),
            row=1, col=2
        )
        
fig.update_layout(
    title_text="Rating and Sentiment Relationship",
    height=400,
    barmode='stack'
    
)

fig.show()

    

In [20]:
category_counts = df['product_category'].value_counts()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=category_counts.values,
    y=category_counts.index,
    orientation='h',
    marker_color='lightcoral',
    text=category_counts.values,
    textposition='auto'
))

fig.update_layout(
    title_text="Product Category Distribution",
    xaxis_title="Number of Reviews",
    yaxis_title="Product Category",
    height=500
)

fig.show()

In [21]:
category_sentiment = pd.crosstab(df['product_category'], df['sentiment'], normalize='index') * 100
category_sentiment = category_sentiment.sort_values('positive', ascending=False)

fig= go.Figure()
for sentiment in ['positive', 'neutral', 'negative']:
    if sentiment in category_sentiment.columns:
        fig.add_trace(go.Bar(
            name=sentiment.capitalize(),
            x=category_sentiment[sentiment].values,
            y=category_sentiment.index,
            orientation='h',
            marker_color=['green', 'blue', 'red'][['positive', 'neutral', 'negative'].index(sentiment)],
            text=category_sentiment[sentiment].round(1).astype(str) + '%',
            textposition='inside'
        ))
        
fig.update_layout(
    title_text="Sentiment Distribution by Product Category",
    xaxis_title="Percentage (%)",
    yaxis_title="Product Category",
    barmode='stack',
    height=600,
    legend_title="Sentiment"
)

fig.show()

In [22]:
country_counts = df['country'].value_counts().head(15)

fig =go.Figure() 
fig.add_trace(go.Bar(
    x=country_counts.index,
    y=country_counts.values,
    marker_color='lightseagreen',
    text=country_counts.values,
    textposition='auto'
))

fig.update_layout(
    title_text="Top 15 Countries by Number of Reviews",
    xaxis_title="Country",
    yaxis_title="Number of Reviews",
    height=400
)
fig.show()


In [23]:
country_sentiment = pd.crosstab(df['country'], df['sentiment'], normalize='index') * 100
country_sentiment = country_sentiment.sort_values('positive', ascending=False)

fig= go.Figure()
for sentiment in ['positive', 'neutral', 'negative']:
    if sentiment in country_sentiment.columns:
        fig.add_trace(go.Bar(
            name=sentiment.capitalize(),
            x=country_sentiment.index[:15],
            y=country_sentiment[sentiment].values[:15],
            marker_color=['green', 'blue', 'red'][['positive', 'neutral', 'negative'].index(sentiment)],
            text=country_sentiment[sentiment].round(1).astype(str) + '%',
            textposition='inside'
        ))
fig.update_layout(
    title_text="Sentiment Distribution by Country (Top 15)",
    yaxis_title="Percentage",
    xaxis_title="Country",
    height=400,
    barmode='stack',
    legend_title="Sentiment"
)
fig.show()

In [24]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return 'unknown'

In [25]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def remove_stopwords(text):
    if not text or text.strip() == "":
        return ""
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

def lemmatize_text(text):
    if not text or text.strip() == "":
        return ""
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

def preprocess_pipeline(text):
    cleaned = clean_text(text)
    no_stopwords = remove_stopwords(cleaned)
    lematized = lemmatize_text(no_stopwords)
    return lematized 

In [26]:
df['language'] = df['review'].apply(detect_language)
print(f'Languages detected: {df["language"].value_counts().to_dict()}')


Languages detected: {'en': 12914, 'fr': 6343, 'de': 5750, 'es': 4541, 'nl': 268, 'pt': 158, 'it': 8, 'af': 8, 'no': 6, 'id': 4}


In [27]:
df['cleaned_review'] = df['review'].apply(clean_text)

In [28]:

df['processed_review'] = df['cleaned_review'].apply(preprocess_pipeline)


In [29]:
sample_indices = [0, 10, 1000]
for idx in sample_indices:
    if idx < len(df):
        print(f"\n Original: {df['review'].iloc[idx][:100]}...")
        print(f"Processed: {df['processed_review'].iloc[idx][:100]}...")


 Original: Absolutely love this product! Best purchase I've made this year....
Processed: absolutely love product best purchase ive made year...

 Original: Extremely poor quality. Looks nothing LIKE the picture. 😍...
Processed: extremely poor quality look nothing like picture...

 Original: Utterly useless product. Straight in the bin....
Processed: utterly useless product straight bin...


In [30]:
final_columns = [
    'review_id', 'review', 'processed_review', 'word_count', 'review_length',
    'rating', 'sentiment', 'product_category', 'country', 'language', 'timestamp'
]

In [31]:
final_columns = [col for col in final_columns if col in df.columns]
df_final = df[final_columns].copy()

In [32]:
df_final.to_csv('../data/processed/processed_reviews.csv', index=False) 

In [33]:
category_mapping = dict(enumerate(df_final['product_category'].unique()))
country_mapping = dict(enumerate(df_final['country'].unique()))

In [34]:
ENCODERS_PATH = '../artifacts/encoders'

In [35]:
import os
os.makedirs('../artifacts/encoders', exist_ok=True)

category_mapping = dict(enumerate(df['product_category'].unique()))
country_mapping = dict(enumerate(df['country'].unique()))

with open('../artifacts/encoders/category_mapping.txt', 'w') as f:
    for idx, cat in category_mapping.items():
        f.write(f"{idx}: {cat}\n")
        
with open('../artifacts/encoders/country_mapping.txt', 'w') as f:
    for idx, country in country_mapping.items():
        f.write(f"{idx}: {country}\n")
    